# Regressão Linear

## Nem sempre quando tivermos montando uma rede neural teremos a mesma estrutura. Dependendo da complexidade/natureza do nosso problemas, podemos ter diferentes arquiteturas para resolver dado problema

### Tentando ser generalista, o que geralmente vai se repetir (no caso da regressão linear):

* **Camada de Saída:** Tem 1 único neurônio (units=1) e NENHUMA função de ativação (ou ativação linear). Isso permite que a rede preveja qualquer valor real contínuo ($-\infty$ a $+\infty$, como preços, notas, temperaturas).

* **Função de Perda (Loss):** Sempre usamos métricas de erro numérico como MSE (Mean Squared Error) ou MAE (Mean Absolute Error).

* **Métrica de Avaliação:** mae ou mse (em vez de accuracy, que só serve para classificação).

### Tentando ser generalista, o que geralmente vai mudar (no caso da regressão linear):

* Quantidade de neurônios nas camadas ocultas (ex: 16, 32, 64).
  
* Quantidade de camadas intermediárias (1 ou 2 são suficientes para a maioria dos problemas tabulares simples).

* Funções de ativação intermediárias (quase sempre relu).


<br>

### Algumas Diferenças entre os problemas Regressão vs. Classificação (tensorflow/Keras):

<br>

| **Componente** | **Problema de Regressão** | **Classificação Binária (0 ou 1)** | **Classificação Multiclasse (ex: 10 dígitos)** |
| --- | --- | --- | --- |
| **Última Camada (`Dense`)** | `Dense(1)` | `Dense(1, activation='sigmoid')` | `Dense(num_classes, activation='softmax')` |
| **Função de Perda (`loss`)** | `'mse'` ou `'mae'` | `'binary_crossentropy'` | `'sparse_categorical_crossentropy'` |
| **Métricas (`metrics`)** | `['mae', 'mse']` | `['accuracy']` | `['accuracy']` |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


I0000 00:00:1787958427.784647  253515 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787958427.818086  253515 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787958428.721553  253515 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


<br>

### Base de Dados

In [2]:
# Carregando o dataset de preços de imóveis
dados = fetch_california_housing(as_frame=True)
X = dados.data[['MedInc', 'HouseAge', 'AveRooms']]  # Renda média, idade do imóvel, média de quartos
y = dados.target  # Preço da casa em centenas de milhares de dólares

In [4]:
# display(X)

# display(y)

0        4.526
1        3.585
2        3.521
3        3.413
4        3.422
         ...  
20635    0.781
20636    0.771
20637    0.923
20638    0.847
20639    0.894
Name: MedHouseVal, Length: 20640, dtype: float64

<br>

### Separação dos dados de Treino e Teste

In [5]:
# Divisão Treino e Teste (lembram Scikit-Learn)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<br>
<br>

### IMPORTANTE PARA REDES NEURAIS: Normalização dos dados!
### Redes neurais aprendem muito melhor se os dados estiverem na mesma escala.


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

<br>
<br>

### Aqui vamos definir qual a estrura da rede para resolver o problema em questão.

In [7]:
# input_shape=[3] porque temos, como vimos acima, 3 colunas de entrada
modelo_regressao = Sequential([
    # Camada Oculta 1: 16 neurônios aprendem relações não-lineares
    Dense(16, activation='relu', input_shape=[3]),
    
    # Camada Oculta 2: 8 neurônios combinam os padrões aprendidos
    Dense(8, activation='relu'),
    
    # CAMADA DE SAÍDA DE REGRESSÃO (pensando no tipo de problema que temos na regressão): 
    # 1 neurônio, sem ativação (produz o valor contínuo do preço)
    Dense(1)
])

W0000 00:00:1787958607.089921  253515 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


<br>

### Compilação do Modelo

In [8]:
modelo_regressao.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='mean_squared_error',  # Penaliza erros grandes (MSE)
    metrics=['mae']             # Erro médio absoluto em $100k pra ficar mais fácil interpretação
)

<br>

### Treinamento
Treinando a Rede Neural de Regressão

In [17]:
historico = modelo_regressao.fit(
    X_train_scaled, y_train,
    validation_split=0.2,       # Separa 20% do treino para validação a cada época
    epochs=100,
    batch_size=64,
    verbose=1
)

Epoch 1/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - loss: 0.5362 - mae: 0.5312 - val_loss: 0.5456 - val_mae: 0.5326
Epoch 2/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step - loss: 0.5365 - mae: 0.5310 - val_loss: 0.5416 - val_mae: 0.5276
Epoch 3/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 674us/step - loss: 0.5373 - mae: 0.5319 - val_loss: 0.5418 - val_mae: 0.5301
Epoch 4/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 631us/step - loss: 0.5373 - mae: 0.5320 - val_loss: 0.5424 - val_mae: 0.5278
Epoch 5/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 639us/step - loss: 0.5380 - mae: 0.5339 - val_loss: 0.5464 - val_mae: 0.5375
Epoch 6/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step - loss: 0.5386 - mae: 0.5325 - val_loss: 0.5440 - val_mae: 0.5328
Epoch 7/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 715us/step - loss: 0.5385 - mae: 0.5328 - val_loss: 0.5470 - val_mae: 0.5428
Epoch 8/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 650us/step - loss: 0.5401 - mae: 0.5333 - val_loss: 0.5452 - val_mae: 0.5416
Epoch 9/100
207/207 ━━━━━━━━━━━━

<br>
<br>

### Podemos avaliar e tentar fazer previsões

In [10]:
# Fazendo predições no conjunto de teste que a rede nunca viu
predicoes = modelo_regressao.predict(X_test_scaled).flatten()

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 324us/step


In [11]:
# Podemos calcular algumas métricas de desempenho
mae = mean_absolute_error(y_test, predicoes)
r2 = r2_score(y_test, predicoes)

In [12]:
print("\nResultados no conj. de teste")
print(f"Erro Médio Absoluto (MAE): ${mae * 100_000:,.2f}")
print(f"R² Score: {r2:.3f}")


Resultados no conj. de teste
Erro Médio Absoluto (MAE): $53,578.02
R² Score: 0.569


In [13]:
# Vamos fazer uma tabela para poder comparar y e y_chapeu para alguns valores
tabela_comparacao = pd.DataFrame({
    'Valor Real ($100k)': y_test.iloc[:10].values,
    'Valor Previsto ($100k)': np.round(predicoes[:10], 2)
})

In [14]:
print("\nAmostra de Previsões:")
tabela_comparacao


Amostra de Previsões:


,Valor Real ($100k),Valor Previsto ($100k)
0,0.47700,1.07
1,0.45800,1.17
2,5.00001,2.86
3,2.18600,2.56
4,2.78000,1.78
5,1.58700,2.06
6,1.98200,2.56
7,1.57500,2.00
8,3.40000,2.38
9,4.46600,4.59


In [15]:
tabela_comparacao['erro'] = np.abs(tabela_comparacao['Valor Real ($100k)'] - tabela_comparacao['Valor Previsto ($100k)'])

In [16]:
tabela_comparacao

,Valor Real ($100k),Valor Previsto ($100k),erro
0,0.47700,1.07,0.59300
1,0.45800,1.17,0.71200
2,5.00001,2.86,2.14001
3,2.18600,2.56,0.37400
4,2.78000,1.78,1.00000
5,1.58700,2.06,0.47300
6,1.98200,2.56,0.57800
7,1.57500,2.00,0.42500
8,3.40000,2.38,1.02000
9,4.46600,4.59,0.12400
